<h1  style="color:#000000; text-align:center; font-weight:bold;">DATASCI 530 Assignment 3</h1>
<h1  style="color:#000000; text-align:center; font-weight:bold;">OpenAlex</h1>
<p  style="color:#000000; text-align:center; font-size:18px;">Maxine Hakimi-Bulson</p>

For  this  assignment,  you  will  interact  with  an  API  and  extract  information  encoded  in  JSON 
format. OpenAlex is a fully open catalog of the global research system, with millions of records 
from different types of sources. You can read more about it here: 
https://docs.openalex.org 
https://docs.openalex.org/quickstart-tutorial 

For this assignment, you will need to use the Works API, which allows you to extract information 
about individual publications and search by topic, country code, etc. Read the page about how 
to filter works: https://docs.openalex.org/api-entities/works 

OpenAlex  has  a  series  of  Python  tutorials  on  how  to  interact  with  the  API.  Use  these  as 
inspiration for  your  own  analysis.  The  tutorial  on  Japanese  publications,  while focusing  on  a 
different set of questions, could be a useful reference. See more details here: 
https://docs.openalex.org/how-to-use-the-api/api-overview 
https://docs.openalex.org/additional-help/tutorials 

For this homework, you are expected to process JSON-formatted data and automate your API 
calls using control flow. 

Deliverable: Submit an HTML file with your results on Canvas. Include Markdown text chunks 
to write comments on your findings and explain your methodology. You are expected to have 
a  title  for  your  project  and  a  short  section  at  the  beginning  summarizing  the  findings.  We 
recommend that you do not print raw data and only print final tables and/or figures. 
When  submitting  your  work,  imagine  that  you  are  sharing  your  report  with  colleagues  who 
have asked you to explain and justify the key decisions and findings from your analysis. The file 
should be professionally presented, self-explanatory, and have clearly labeled sections. 

<h2 style="color:#000000;">OpenAlex</h2>

write a couple sentences about OpenAlex here

<h2 style="color:#000000;">Summary of Findings</h2>

write summary of findings here in a bulleted list like this:

Here are the main findings:

- 1

- 2

- 3

- 4

- 5

<h2 style="color:#000000;">Import Packages</h2>

First we will import the necessary packages to complete our analysis.

In [1]:
import requests      # Requests data from an API
import json          # Handles JSON object
import jmespath      # Allows you to mor easily navigate JSON objects
import pandas as pd  # Handles data frames

<h2 style="color:#000000;">Inquiry & Analysis</h2>

<h3 style="color:#000000;"> 1. How many works have “artificial intelligence” in their titles?  </h3>

I sent a request to OpenAlex Works API using title.search to restrict matches to work titles containing the phrase "artificial intelligence." I read meta["count"] from the JSON response to get the total count for titles with the phrase "artificial intelligence." As a quality check, I inspected the response keys and a returned title to confirm the response structure and search relevance.

In [13]:
# url to search for titles that contain the phrase "artificial intelligence" in the title
url_openalex_ai = 'https://api.openalex.org/works?filter=title.search:"artificial intelligence"'

# send the request for the title search data
search_results_ai = requests.get(url_openalex_ai).json()

# browse keys in the first level
#print(search_results_ai.keys())

# use meta["count"] as it gives the number of all matching works ("results" only gives first page)
print(search_results_ai["meta"]["count"])

# quality check: confirm that "artificial intelligence" is in the title of one
#print(search_results_ai["results"][0]["title"])

330584


There are 330,584 works with "artificial intelligence" in their titles.

<h3 style="color:#000000;"> 2. Restrict  the  analysis  to  works  published  since  2000  and  group  by  the  country  of  the 
authors’  affiliations.  Produce  a  table  that  sorts  the  countries  by  the  number  of 
publications about artificial intelligence.   </h3>

I kept the title search from (1) and added "from_publication_date:2000-01-01" to filter to works published since 2000. I grouped the works by countries of their authors’ affiliations. The first response reached OpenAlex’s 200 group page limit, so I used cursor pagination and a while loop to collect all 212 country groups. I printed a table of my findings, sorting the count in descending order and displayed the countries with the most publications with "artificial intelligence" in the title since 2000.

In [31]:
# url to search for titles that contain the phrase "artificial intelligence" in the title
# and filter to only include works published since 2000
url_openalex_ai_2000 = \
    'https://api.openalex.org/works?filter=title.search:"artificial intelligence",from_publication_date:2000-01-01'

# quality check: confirm the search is including the 2000 limit
#print(url_openalex_ai_2000)

# group by the country of the authors' affiliations
url_openalex_ai_2000_grouped = url_openalex_ai_2000 + '&group_by=authorships.countries'

# quality check: confirm the search groups by country
#print(url_openalex_ai_2000_grouped)
#first_response_ai_2000 = requests.get(url_openalex_ai_2000_grouped).json()
#print("Groups in first response:", len(first_response_ai_2000["group_by"]))
#print("Groups reported for that page:", first_response_ai_2000["meta"]["groups_count"])

# the first response contains 200 groups, the maximum per page
# use cursor pagination to collect all remaining groups
country_groups_ai_2000 = []
cursor_ai_2000 = "*"
groups_per_page_ai_2000 = []

while cursor_ai_2000 is not None:
    page_ai_2000 = requests.get(
        url_openalex_ai_2000_grouped,
        params={"cursor": cursor_ai_2000}
    ).json()

    groups_on_page = page_ai_2000["group_by"]
    country_groups_ai_2000.extend(groups_on_page)
    groups_per_page_ai_2000.append(len(groups_on_page))
    cursor_ai_2000 = page_ai_2000["meta"]["next_cursor"]

# quality check: show pagination and the total number collected
#print(groups_per_page_ai_2000)
#print(len(country_groups_ai_2000))

# make results table
table_ai_2000_grouped = pd.DataFrame(country_groups_ai_2000)

# quality check: inspect the variables
#print(table_ai_2000_grouped.columns)

table_ai_2000_grouped = table_ai_2000_grouped[
    ["key_display_name", "count"]
]
# rename columns
table_ai_2000_grouped = table_ai_2000_grouped.rename(
    columns={
        "key_display_name": "Country",
        "count": "Publications"
    }
)
# sort by number of publications
table_ai_2000_grouped = table_ai_2000_grouped.sort_values(
    by="Publications",
    ascending=False
).reset_index(drop=True)

# display results table
display(
    table_ai_2000_grouped.head(20)
    .style
    .format({"Publications": "{:,.0f}"})
    .set_caption("Top 20 countries by publications with 'artificial intelligence' in the title (Since 2000)")
    .hide(axis="index")
)


Country,Publications
United States,"40,115"
China,"29,918"
India,"26,366"
United Kingdom,"13,127"
Indonesia,"9,082"
Italy,"7,407"
Germany,"6,939"
Turkey,"6,591"
Canada,"6,345"
Russia,"6,042"


Since 2000, authors from the United States have produced the most publications on artificial intelligence. 

<h3 style="color:#000000;"> 3. What are the titles of the 10 most highly cited papers on artificial intelligence since 2000? 
How  many  citations  do  they  have?  Who  are  the  authors?  What  institutions  are  they 
affiliated with?   </h3>

explain methods here

The titles of the 10 most highly cited papers on artificial intellegece since 2000 are:

Number of Citations:

Authors:

Institutions:

<h3 style="color:#000000;"> 4. How  has  the  number  of  publications  with  the  phrase  “artificial intelligence” changed 
year-to-year since 2000? How does this compare to the use of the terms “LLM” and 
“GPT”? Repeat the year-to-year analysis for another keyword of your choice. Plot your 
results in a graph.   </h3>

explain methods here

Since 2000, the number of publications with the phrase "artificial intelligence" has ___

Comparing this to the use of the terms "LLM" and "GPT", ___

Comparing this to the keyword "____", _____

<h2 style="color:#000000;">New Insights</h2>

<h3 style="color:#000000;"> 5. Compute an additional table or figure not included in the questions above.  </h3>

explain methods here

<h3 style="color:#000000;"> 6. What do we learn from this new table or figure?  </h3>

answer here